In [1]:
import os

import pandas as pd
import torch
from PIL import Image
from torch.backends import cudnn
from torch.utils.data import DataLoader
from torchvision.datasets import VisionDataset
from torchvision.transforms import InterpolationMode, v2

torch.manual_seed(3407)
if torch.cuda.is_available():
    torch.cuda.manual_seed(3407)

cudnn.deterministic = True
cudnn.benchmark = False
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)


# --- Speed: avoid CPU oversubscription (common cause of dataloader slowdown/timeouts).
# Preserves correctness; only affects performance/parallel scheduling.
try:
    torch.set_num_threads(max(1, min(4, os.cpu_count() or 4)))
    torch.set_num_interop_threads(1)
except Exception:
    pass


def _resolve_path(p: str) -> str:
    if os.path.exists(p):
        return p
    if p.startswith("/kaggle/input/"):
        alt = p.replace("/kaggle/input/", "/kaggle/data/")
        if os.path.exists(alt):
            return alt
    return p


def _resolve_images_dir(images_path: str) -> str:
    images_path = _resolve_path(images_path)

    if os.path.isfile(images_path):
        images_path = os.path.dirname(images_path)

    candidates = [
        images_path,
        os.path.join(images_path, "train_images"),
        os.path.join(images_path, "test_images"),
        os.path.join(images_path, "train_images", "train_images"),
        os.path.join(images_path, "test_images", "test_images"),
    ]

    exts = (".jpg", ".jpeg", ".png", ".bmp", ".webp", ".tif", ".tiff")

    def has_images(d: str) -> bool:
        if not os.path.isdir(d):
            return False
        try:
            for n in os.listdir(d):
                fp = os.path.join(d, n)
                if os.path.isfile(fp) and n.lower().endswith(exts):
                    return True
        except Exception:
            return False
        return False

    for c in candidates:
        if has_images(c):
            return c

    for c in candidates:
        if os.path.isdir(c):
            return c

    return images_path


test_dir = _resolve_images_dir(
    "/kaggle/input/cassava-leaf-disease-classification/test_images/"
)
train_dir = _resolve_images_dir(
    "/kaggle/input/cassava-leaf-disease-classification/train_images/"
)

train_csv_path = _resolve_path(
    "/kaggle/input/cassava-leaf-disease-classification/train.csv"
)
sample_sub_path = _resolve_path(
    "/kaggle/input/cassava-leaf-disease-classification/sample_submission.csv"
)

eff_img_size = 528
vit_img_size = 224

batch_size = 16
num_workers = 4
num_classes = 5
tta = True

do_fallback_head_train = True
fallback_epochs = 6
fallback_lr = 1e-3


def _safe_torch_load(path, map_location):
    path = _resolve_path(path)
    try:
        if path and os.path.exists(path):
            return torch.load(path, map_location=map_location)
    except Exception:
        return None
    return None


def _as_module_or_none(obj):
    import torch.nn as nn

    if obj is None:
        return None, None
    if isinstance(obj, nn.Module):
        return obj, None
    if isinstance(obj, dict):
        return None, obj
    return None, None


vit_obj = _safe_torch_load("/kaggle/input/vit-v1-update/vit_v1_1.pt", device)
eff_obj = _safe_torch_load("/kaggle/input/efficient-net/vit_cont_3.pt", device)
head_obj = _safe_torch_load("/kaggle/input/linear-cls/linear_cls.pt", device)

vit_model, vit_sd = _as_module_or_none(vit_obj)
eff_model, eff_sd = _as_module_or_none(eff_obj)
linear_head, head_sd = _as_module_or_none(head_obj)

using_fallback_backbones = False
if vit_model is None or eff_model is None:
    using_fallback_backbones = True
    import torch.nn as nn
    import torchvision

    vit_model = torchvision.models.vit_b_16(
        weights=torchvision.models.ViT_B_16_Weights.DEFAULT
    )
    vit_model.heads = nn.Identity()

    eff_model = torchvision.models.efficientnet_b0(
        weights=torchvision.models.EfficientNet_B0_Weights.DEFAULT
    )
    eff_model.classifier = nn.Identity()

vit_model.eval()
eff_model.eval()
with torch.no_grad():
    dummy_vit = torch.zeros(1, 3, vit_img_size, vit_img_size)
    dummy_eff = torch.zeros(1, 3, eff_img_size, eff_img_size)
    vit_dim = vit_model(dummy_vit).shape[1]
    eff_dim = eff_model(dummy_eff).shape[1]

if linear_head is None:
    import torch.nn as nn

    linear_head = nn.Linear(vit_dim + eff_dim, num_classes)

if head_sd is not None:
    try:
        linear_head.load_state_dict(head_sd, strict=True)
    except Exception:
        pass

if vit_sd is not None:
    try:
        vit_model.load_state_dict(vit_sd, strict=False)
    except Exception:
        pass
if eff_sd is not None:
    try:
        eff_model.load_state_dict(eff_sd, strict=False)
    except Exception:
        pass

vit_model = vit_model.to(device).eval()
eff_model = eff_model.to(device).eval()
linear_head = linear_head.to(device).eval()




cuda


Downloading: "https://download.pytorch.org/models/vit_b_16-c867db91.pth" to /root/.cache/torch/hub/checkpoints/vit_b_16-c867db91.pth
100%|██████████| 330M/330M [00:03<00:00, 109MB/s]
Downloading: "https://download.pytorch.org/models/efficientnet_b0_rwightman-7f5810bc.pth" to /root/.cache/torch/hub/checkpoints/efficientnet_b0_rwightman-7f5810bc.pth
100%|██████████| 20.5M/20.5M [00:00<00:00, 71.4MB/s]


In [2]:
class CassavaDataset(VisionDataset):
    """Custom dataset for the Cassava data.

    Args:
        data_dir: base directory to the images.
        transforms: set of transforms to be used.
        image_ids: optional explicit list of image filenames to use (e.g., from CSV).
    """

    def __init__(
        self,
        data_dir,
        vit_size,
        efficient_size,
        vit_transform=None,  # allow per-backbone normalization
        eff_transform=None,  # allow per-backbone normalization
        ttas=None,
        img_size=384,
        labels_df=None,  # optional (for training head fallback)
        image_ids=None,  # optional explicit list of filenames
    ):
        data_dir = _resolve_path(data_dir)
        exts = (".jpg", ".jpeg", ".png", ".bmp", ".webp", ".tif", ".tiff")

        def list_images(d):
            if not os.path.isdir(d):
                return []
            names = os.listdir(d)
            return sorted(
                [
                    n
                    for n in names
                    if os.path.isfile(os.path.join(d, n)) and n.lower().endswith(exts)
                ]
            )

        imgs_here = list_images(data_dir)
        if len(imgs_here) == 0:
            base = data_dir
            candidates = [
                os.path.join(base, "train_images"),
                os.path.join(base, "test_images"),
                os.path.join(base, os.path.basename(base)),
            ]
            for c in candidates:
                imgs_c = list_images(c)
                if len(imgs_c) > 0:
                    data_dir = c
                    imgs_here = imgs_c
                    break

        super().__init__(root=data_dir)

        self.vit_transform = vit_transform
        self.eff_transform = eff_transform
        self.labels_df = labels_df

        if image_ids is not None:
            imgs = []
            for n in image_ids:
                fp = os.path.join(self.root, n)
                if os.path.isfile(fp) and n.lower().endswith(exts):
                    imgs.append(n)
            self.images = imgs
        else:
            self.images = imgs_here

        self.ttas = ttas

        # --- Speed: do a single PIL->Tensor conversion and perform crop/resize on tensors.
        # This preserves the same operations (center crop to 600x600, then resize to two sizes)
        # but avoids doing PIL operations twice (once per backbone) and reduces Python overhead.
        self.cc = v2.CenterCrop((600, 600))
        self.resize_vit = v2.Resize(
            (vit_size, vit_size),
            interpolation=InterpolationMode.BICUBIC,
            antialias=True,
        )
        self.resize_efficient = v2.Resize(
            (efficient_size, efficient_size),
            interpolation=InterpolationMode.BICUBIC,
            antialias=True,
        )
        self.to_image = v2.ToImage()  # yields torch uint8 CHW
        self.to_float = v2.ToDtype(torch.float32, scale=True)

        if self.labels_df is not None:
            self._label_map = dict(
                zip(self.labels_df["image_id"].values, self.labels_df["label"].values)
            )
        else:
            self._label_map = None

    @staticmethod
    def _hflip(x: torch.Tensor) -> torch.Tensor:
        return torch.flip(x, dims=(-1,))

    @staticmethod
    def _vflip(x: torch.Tensor) -> torch.Tensor:
        return torch.flip(x, dims=(-2,))

    def __getitem__(self, idx):
        filename = self.images[idx]
        img = Image.open(os.path.join(self.root, filename)).convert("RGB")

        # One crop on PIL (fast) then convert once; resizing on tensor for both backbones.
        img = self.cc(img)
        img_t = self.to_image(img)  # uint8 CHW

        vit_img = self.resize_vit(img_t)
        eff_img = self.resize_efficient(img_t)

        if self.ttas is not None:
            # --- Speed: vectorize TTAs without per-tta PIL work.
            # Semantics identical: [HFlip, VFlip, Identity] applied to the *resized* images.
            vit_imgs = [self._hflip(vit_img), self._vflip(vit_img), vit_img]
            eff_imgs = [self._hflip(eff_img), self._vflip(eff_img), eff_img]

            if self.vit_transform is not None:
                vit_imgs = [self.vit_transform(x) for x in vit_imgs]
            if self.eff_transform is not None:
                eff_imgs = [self.eff_transform(x) for x in eff_imgs]
            vit_img, eff_img = vit_imgs, eff_imgs
        else:
            if self.vit_transform is not None:
                vit_img = self.vit_transform(vit_img)
            if self.eff_transform is not None:
                eff_img = self.eff_transform(eff_img)

        if self._label_map is None:
            return vit_img, eff_img, filename
        else:
            y = int(self._label_map[filename])
            return vit_img, eff_img, y, filename

    def __len__(self):
        return len(self.images)




In [3]:
# --- Speed: keep transforms but make them tensor-native (no repeated ToImage/ToDtype work).
# Correctness preserved: input to Normalize remains float32 in [0,1], same as before.
vit_transforms = v2.Compose(
    [
        v2.ToDtype(torch.float32, scale=True),
        v2.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5]),  # ViT_B_16 default
    ]
)

eff_transforms = v2.Compose(
    [
        v2.ToDtype(torch.float32, scale=True),
        v2.Normalize(
            mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]
        ),  # EfficientNet default
    ]
)

# TTAs are implemented inside dataset (as exact hflip/vflip/identity); keep tta flag semantics.
ttas = [True] if tta else None  # sentinel: dataset checks non-None for TTA

test_dataset = CassavaDataset(
    test_dir,
    vit_img_size,
    eff_img_size,
    vit_transform=vit_transforms,
    eff_transform=eff_transforms,
    ttas=ttas,
)

# --- Speed: improve DataLoader throughput with persistent workers + prefetching.
# Does not change samples/ordering/values.
_loader_kwargs = dict(
    batch_size=batch_size,
    shuffle=False,
    num_workers=num_workers,
    pin_memory=torch.cuda.is_available(),
)
if num_workers > 0:
    _loader_kwargs.update(
        dict(
            persistent_workers=True,
            prefetch_factor=4,
        )
    )

test_loader = DataLoader(test_dataset, **_loader_kwargs)

normalizer = torch.nn.Softmax(dim=1)



In [4]:
need_head_training = (head_sd is None) and do_fallback_head_train

if need_head_training:
    import torch.nn as nn

    train_df = pd.read_csv(train_csv_path)
    train_image_ids = train_df["image_id"].astype(str).tolist()

    train_dataset = CassavaDataset(
        train_dir,
        vit_img_size,
        eff_img_size,
        vit_transform=vit_transforms,
        eff_transform=eff_transforms,
        ttas=None,
        labels_df=train_df,
        image_ids=train_image_ids,
    )

    if len(train_dataset) == 0:
        raise RuntimeError(
            f"Training dataset is empty after filtering. train_dir={train_dir} (resolved root={train_dataset.root})"
        )

    _train_loader_kwargs = dict(
        batch_size=batch_size,
        shuffle=True,
        num_workers=num_workers,
        pin_memory=torch.cuda.is_available(),
    )
    if num_workers > 0:
        _train_loader_kwargs.update(
            dict(
                persistent_workers=True,
                prefetch_factor=4,
            )
        )

    train_loader = DataLoader(train_dataset, **_train_loader_kwargs)

    for p in vit_model.parameters():
        p.requires_grad = False
    for p in eff_model.parameters():
        p.requires_grad = False
    for p in linear_head.parameters():
        p.requires_grad = True

    vit_model.eval()
    eff_model.eval()
    linear_head.train()

    opt = torch.optim.AdamW(linear_head.parameters(), lr=fallback_lr)
    criterion = nn.CrossEntropyLoss()

    for epoch in range(fallback_epochs):
        for vit_inputs, eff_inputs, y, _fn in train_loader:
            vit_inputs = vit_inputs.to(device, non_blocking=True)
            eff_inputs = eff_inputs.to(device, non_blocking=True)
            y = y.to(device, non_blocking=True)

            with torch.no_grad():
                vit_feats = vit_model(vit_inputs)
                eff_feats = eff_model(eff_inputs)
                feats = torch.cat([vit_feats, eff_feats], dim=1)

            logits = linear_head(feats)
            loss = criterion(logits, y)

            opt.zero_grad(set_to_none=True)
            loss.backward()
            opt.step()

    linear_head.eval()



In [5]:
all_names = []
all_preds = []

vit_model.eval()
eff_model.eval()
linear_head.eval()

with torch.no_grad():
    for batch_idx, (vit_inputs, eff_inputs, filenames) in enumerate(test_loader):
        bs = len(filenames)

        if tta:
            # vit_inputs/eff_inputs are lists length T, each [B,C,H,W] due to dataset output.
            # --- Speed: avoid split/stack overhead; reshape after cat (equivalent averaging).
            vit_inputs = torch.cat(vit_inputs, dim=0).to(
                device, non_blocking=True
            )  # [T*B,...]
            eff_inputs = torch.cat(eff_inputs, dim=0).to(device, non_blocking=True)
            filenames = list(filenames)

            vit_outputs = vit_model(vit_inputs)  # [T*B, Dv]
            eff_outputs = eff_model(eff_inputs)  # [T*B, De]

            T = vit_outputs.shape[0] // bs
            vit_mean_logits = vit_outputs.view(T, bs, -1).mean(dim=0)
            eff_mean_logits = eff_outputs.view(T, bs, -1).mean(dim=0)

            logit_inputs = torch.cat([vit_mean_logits, eff_mean_logits], dim=1)
            outputs = linear_head(logit_inputs)

            mean_preds = normalizer(outputs)
            pred_labels = torch.argmax(mean_preds, 1).tolist()
        else:
            vit_inputs = vit_inputs.to(device, non_blocking=True)
            eff_inputs = eff_inputs.to(device, non_blocking=True)
            filenames = list(filenames)

            vit_outputs = vit_model(vit_inputs)
            eff_outputs = eff_model(eff_inputs)

            logit_inputs = torch.cat([vit_outputs, eff_outputs], dim=1)
            outputs = linear_head(logit_inputs)

            preds = normalizer(outputs)
            pred_labels = torch.argmax(preds, 1).tolist()

        all_names.extend(filenames)
        all_preds.extend(pred_labels)



In [6]:
sample_sub = pd.read_csv(sample_sub_path)
pred_map = dict(zip(all_names, all_preds))

sample_sub["label"] = sample_sub["image_id"].map(pred_map)
sample_sub["label"] = sample_sub["label"].fillna(0).astype(int)

sample_sub.to_csv("submission.csv", index=False)
sample_sub

,image_id,label
0,1234294272.jpg,3
1,1234332763.jpg,3
2,1234375577.jpg,1
3,1234555380.jpg,1
4,1234571117.jpg,3
...,...,...
2671,2656351084.jpg,3
2672,2656378111.jpg,3
2673,2656781269.jpg,3
2674,2656964450.jpg,3
